In [4]:
# Cell 1 — Imports + configuration

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root = parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent

# Add src/ to Python path
sys.path.append(str(PROJECT_ROOT / "src"))

from swat_gnn.data.loader import (
    load_config,
    load_training_data,
    load_validation_data,
    load_test_data,
)

# Give the loader the absolute path to the config file
CONFIG_PATH = PROJECT_ROOT / "configs" / "data.yaml"

config = load_config(str(CONFIG_PATH))

print("Project root:", PROJECT_ROOT)
print("Dataset:", config["dataset"]["name"])
print("Root:", config["dataset"]["root"])

Project root: c:\Users\adima\Projects\industrial-dynamic-graph-anomaly-detection
Dataset: HAI_23.05
Root: data/raw/HAI_Kaggle/hai-23.05


In [8]:
# Cell 2 — Load the dataset

training_segments = load_training_data(config)
validation_segments = load_validation_data(config)
test_sensors, test_labels = load_test_data(config)

print("Training segments:", len(training_segments))
print("Validation segments:", len(validation_segments))
print("Test segments:", len(test_sensors))

for i, df in enumerate(training_segments, 1):
    print(f"Train {i}: {df.shape}")

print(f"Validation: {validation_segments[0].shape}")

for i, df in enumerate(test_sensors, 1):
    print(f"Test {i}: {df.shape}, labels: {test_labels[i-1].shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\HAI_Kaggle\\hai-23.05\\hai-train1.csv'

In [ ]:
# Check timestamp integrity

datasets = [
    ("train1", training_segments[0]),
    ("train2", training_segments[1]),
    ("train3", training_segments[2]),
    ("validation", validation_segments[0]),
    ("test1", test_sensors[0]),
    ("test2", test_sensors[1]),
]

for name, df in datasets:
    timestamps = df["timestamp"]

    time_diff = timestamps.diff().dropna()

    print(f"\n{name}")
    print("-" * 50)
    print("Start:", timestamps.iloc[0])
    print("End:  ", timestamps.iloc[-1])
    print("Rows:", len(df))
    print("Duplicate timestamps:", timestamps.duplicated().sum())
    print("Missing timestamps:", timestamps.isna().sum())
    print("Most common intervals:")
    print(time_diff.value_counts().head(3))

In [ ]:
# Check missing and infinite sensor values

for name, df in datasets:
    sensor_data = df.drop(columns="timestamp")

    missing = sensor_data.isna().sum().sum()
    infinite = np.isinf(sensor_data.to_numpy()).sum()

    print(
        f"{name}: "
        f"missing={missing}, "
        f"infinite={infinite}"
    )

In [ ]:
# Verify that all datasets contain the same sensor variables

reference_features = [
    c for c in training_segments[0].columns
    if c != "timestamp"
]

print("Number of sensor variables:", len(reference_features))

for name, df in datasets:
    features = [c for c in df.columns if c != "timestamp"]

    print(
        f"{name}:",
        "MATCH" if features == reference_features else "MISMATCH"
    )

In [ ]:
# Basic statistical summary of the training data

train1 = training_segments[0]

sensor_columns = [
    c for c in train1.columns
    if c != "timestamp"
]

summary = train1[sensor_columns].describe().T

summary.head(20)

In [ ]:
# Identify constant variables across all training data

all_training_values = pd.concat(
    [df[sensor_columns] for df in training_segments],
    axis=0,
    ignore_index=True,
)

training_std = all_training_values.std()

constant_features = training_std[training_std == 0].index.tolist()

print("Total features:", len(sensor_columns))
print("Constant features:", len(constant_features))

print("\nConstant features:")
for feature in constant_features:
    print("-", feature)

In [ ]:
# Identify near-constant variables

near_constant = training_std.sort_values().head(30)

near_constant

In [ ]:
# Examine feature ranges

feature_ranges = pd.DataFrame({
    "min": all_training_values.min(),
    "max": all_training_values.max(),
    "mean": all_training_values.mean(),
    "std": all_training_values.std(),
})

feature_ranges["range"] = (
    feature_ranges["max"] - feature_ranges["min"]
)

feature_ranges.sort_values("range", ascending=False).head(20)

In [ ]:
# Distribution of sensor standard deviations

plt.figure(figsize=(10, 5))

plt.hist(
    training_std,
    bins=30,
)

plt.xlabel("Feature standard deviation")
plt.ylabel("Number of features")
plt.title("Distribution of Feature Variability")

plt.tight_layout()
plt.show()

In [ ]:
# Compute feature correlations

correlation_matrix = all_training_values.corr()

print("Correlation matrix shape:", correlation_matrix.shape)

correlation_matrix.head()

In [ ]:
# Visualize the correlation matrix

plt.figure(figsize=(14, 12))

plt.imshow(
    correlation_matrix,
    aspect="auto",
    interpolation="nearest",
)

plt.colorbar(label="Correlation")

plt.title("Sensor Correlation Matrix")
plt.xlabel("Sensor")
plt.ylabel("Sensor")

plt.tight_layout()
plt.show()

In [ ]:
# Find strongly correlated sensor pairs

corr = correlation_matrix.abs()

upper_triangle = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

strong_pairs = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

strong_pairs.head(30)

In [ ]:
# Compare feature means across dataset segments

segment_means = pd.DataFrame({
    "train1": training_segments[0][sensor_columns].mean(),
    "train2": training_segments[1][sensor_columns].mean(),
    "train3": training_segments[2][sensor_columns].mean(),
    "validation": validation_segments[0][sensor_columns].mean(),
})

segment_means.head(20)

In [ ]:
# Visualize the distribution of feature means across segments

plt.figure(figsize=(12, 6))

segment_means.boxplot()

plt.ylabel("Feature mean")
plt.title("Feature Mean Distribution Across Dataset Segments")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 — Attack statistics

for i, labels in enumerate(test_labels, 1):
    counts = labels.value_counts().sort_index()

    normal_count = counts.get(0, 0)
    attack_count = counts.get(1, 0)
    total_count = len(labels)

    print(f"\nTest {i}")
    print("-" * 50)
    print("Total timesteps:", total_count)
    print("Normal timesteps:", normal_count)
    print("Attack timesteps:", attack_count)
    print(
        "Attack percentage:",
        round(100 * attack_count / total_count, 2),
        "%"
    )

In [ ]:
# Cell 17 — Identify contiguous attack events

def find_attack_events(labels):
    labels = np.asarray(labels)

    attack_indices = np.where(labels == 1)[0]

    if len(attack_indices) == 0:
        return []

    events = []

    start = attack_indices[0]
    previous = attack_indices[0]

    for index in attack_indices[1:]:
        if index != previous + 1:
            events.append((start, previous))
            start = index

        previous = index

    events.append((start, previous))

    return events


for i, labels in enumerate(test_labels, 1):
    events = find_attack_events(labels)

    durations = [
        end - start + 1
        for start, end in events
    ]

    print(f"\nTest {i}")
    print("-" * 50)
    print("Number of attack events:", len(events))
    print("Total attack timesteps:", sum(durations))

    if durations:
        print("Minimum duration:", min(durations), "seconds")
        print("Maximum duration:", max(durations), "seconds")
        print("Median duration:", np.median(durations), "seconds")

In [ ]:
# Cell 18 — Visualize attack timelines

for i, (df, labels) in enumerate(
    zip(test_sensors, test_labels), 1
):
    plt.figure(figsize=(14, 3))

    plt.plot(
        df["timestamp"],
        labels,
    )

    plt.xlabel("Time")
    plt.ylabel("Attack label")
    plt.title(f"HAI Test {i} Attack Timeline")

    plt.yticks([0, 1], ["Normal", "Attack"])

    plt.tight_layout()
    plt.show()

## 2. Relationship Dynamics: Normal vs. Attack Behavior

This section investigates whether relationships between industrial
variables change during anomalous events.

The analysis is exploratory and is not used for model training,
hyperparameter tuning, or final test-set optimization.

In [2]:
# Cell 20 — Separate normal and attack observations

test1_data = test_sensors[0].drop(columns="timestamp").copy()
test1_labels = test_labels[0].to_numpy()

normal_data = test1_data[test1_labels == 0]
attack_data = test1_data[test1_labels == 1]

print("Test 1 total observations:", len(test1_data))
print("Normal observations:", len(normal_data))
print("Attack observations:", len(attack_data))

print("\nNormal shape:", normal_data.shape)
print("Attack shape:", attack_data.shape)

NameError: name 'test_sensors' is not defined

In [ ]:
# Cell 21 — Correlation matrices for normal and attack periods

normal_correlation = normal_data.corr()
attack_correlation = attack_data.corr()

print("Normal correlation matrix:", normal_correlation.shape)
print("Attack correlation matrix:", attack_correlation.shape)

In [ ]:
# Cell 22 — Measure correlation changes

correlation_change = (
    attack_correlation - normal_correlation
).abs()

print("Average absolute correlation change:",
      correlation_change.values.mean())

print("Maximum absolute correlation change:",
      correlation_change.values.max())

In [1]:
# Cell 23 — Strongest changes in sensor relationships

change = correlation_change.copy()

upper_triangle = change.where(
    np.triu(np.ones(change.shape), k=1).astype(bool)
)

largest_changes = (
    upper_triangle
    .stack()
    .sort_values(ascending=False)
)

largest_changes.head(30)

NameError: name 'correlation_change' is not defined